# Supplier Risk Intelligence System - Analysis Notebook

## End-to-End Analysis: Data Collection → Risk Scoring → Insights

This notebook demonstrates the complete pipeline for supplier risk assessment

## 1. Setup & Data Collection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Libraries imported successfully")

In [ ]:
# Import our custom modules
import sys
sys.path.insert(0, '.')

from data_collector import DataCollector, NewsCollector, FinancialDataCollector
from risk_scoring import RiskScoringEngine, SentimentAnalyzer

print("✅ Custom modules loaded")

In [ ]:
# Step 1: Collect Data
print("🔄 Collecting supplier data...")
collector = DataCollector()
news_df, financial_df = collector.collect_all_data()

print(f"\n📰 News Articles Collected: {len(news_df)}")
print(f"💼 Financial Data Points: {len(financial_df)}")
print(f"📅 Data Collection Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Data Exploration

In [ ]:
# Explore news data
print("\n=== NEWS DATA OVERVIEW ===")
print(news_df.head())
print(f"\nDate Range: {news_df['published'].min()} to {news_df['published'].max()}")
print(f"\nNews Sources:")
print(news_df['source'].value_counts())

In [ ]:
# Companies mentioned in news
print("\n=== COMPANIES IN NEWS ===")
all_companies = []
for companies_str in news_df['companies'].dropna():
    all_companies.extend([c.strip() for c in companies_str.split(',')])

company_mentions = pd.Series(all_companies).value_counts()
print(company_mentions.head(15))

# Visualize company mentions
fig, ax = plt.subplots(figsize=(12, 6))
company_mentions.head(10).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of News Mentions')
ax.set_title('Top 10 Suppliers in News Coverage (Last 7 Days)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Explore financial data
print("\n=== FINANCIAL DATA OVERVIEW ===")
print(financial_df.head(10))
print(f"\nStatistical Summary:")
print(financial_df[['volatility', 'price_trend']].describe().round(2))

## 3. Sentiment Analysis

In [ ]:
# Analyze sentiment of news articles
print("\n=== SENTIMENT ANALYSIS ===")
analyzer = SentimentAnalyzer()

sentiments = []
for idx, row in news_df.iterrows():
    text = str(row['title']) + ' ' + str(row['summary'])
    sentiment = analyzer.analyze_sentiment(text)
    sentiments.append(sentiment)

news_df['sentiment'] = sentiments

print(f"\nAverage Sentiment: {news_df['sentiment'].mean():.3f}")
print(f"Sentiment Std Dev: {news_df['sentiment'].std():.3f}")
print(f"\nSentiment Distribution:")
print(f"  Positive (> 0.3): {(news_df['sentiment'] > 0.3).sum()} articles")
print(f"  Neutral (-0.3 to 0.3): {((news_df['sentiment'] >= -0.3) & (news_df['sentiment'] <= 0.3)).sum()} articles")
print(f"  Negative (< -0.3): {(news_df['sentiment'] < -0.3).sum()} articles")

# Visualize sentiment
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1.hist(news_df['sentiment'], bins=20, color='skyblue', edgecolor='black', alpha=0.7)
ax1.axvline(news_df['sentiment'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
ax1.set_xlabel('Sentiment Score')
ax1.set_ylabel('Number of Articles')
ax1.set_title('Distribution of News Sentiment', fontweight='bold')
ax1.legend()

# Box plot by sentiment category
sentiment_categories = []
for s in news_df['sentiment']:
    if s > 0.3:
        sentiment_categories.append('Positive')
    elif s < -0.3:
        sentiment_categories.append('Negative')
    else:
        sentiment_categories.append('Neutral')

sentiment_counts = pd.Series(sentiment_categories).value_counts()
sentiment_counts.plot(kind='bar', ax=ax2, color=['#ff7f0e', '#2ca02c', '#d62728'])
ax2.set_title('Article Sentiment Categories', fontweight='bold')
ax2.set_ylabel('Count')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

## 4. Risk Scoring

In [ ]:
# Calculate risk scores
print("\n=== RISK SCORING ENGINE ===")
print("Calculating composite risk scores for all suppliers...\n")

scoring_engine = RiskScoringEngine()
risk_scores = scoring_engine.score_suppliers(news_df, financial_df)

print(risk_scores.to_string(index=False))

## 5. Risk Visualization

In [ ]:
# Risk distribution
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# 1. Top suppliers by risk
top_risk = risk_scores.head(10)
colors = ['#d62728' if x >= 75 else '#ff7f0e' if x >= 60 else '#ffd700' if x >= 40 else '#2ca02c' 
          for x in top_risk['risk_score']]
ax1.barh(top_risk['company'], top_risk['risk_score'], color=colors)
ax1.set_xlabel('Risk Score')
ax1.set_title('Top 10 Suppliers by Risk Score', fontweight='bold')
ax1.set_xlim(0, 100)
for i, v in enumerate(top_risk['risk_score']):
    ax1.text(v + 2, i, f'{v:.1f}', va='center')

# 2. Risk score distribution
ax2.hist(risk_scores['risk_score'], bins=15, color='skyblue', edgecolor='black', alpha=0.7)
ax2.axvline(risk_scores['risk_score'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {risk_scores["risk_score"].mean():.1f}')
ax2.set_xlabel('Risk Score')
ax2.set_ylabel('Number of Suppliers')
ax2.set_title('Overall Risk Distribution', fontweight='bold')
ax2.legend()

# 3. News vs Financial Risk
ax3.scatter(risk_scores['financial_risk'], risk_scores['news_risk'], s=150, alpha=0.6, color='steelblue')
for idx, row in risk_scores.head(5).iterrows():
    ax3.annotate(row['company'], (row['financial_risk'], row['news_risk']), fontsize=8)
ax3.set_xlabel('Financial Risk Score')
ax3.set_ylabel('News Risk Score')
ax3.set_title('Financial vs News Risk Factors', fontweight='bold')
ax3.grid(True, alpha=0.3)

# 4. Risk level breakdown
risk_level_counts = risk_scores['risk_level'].value_counts()
colors_pie = {'🔴 CRITICAL': '#d62728', '🟠 HIGH': '#ff7f0e', '🟡 MEDIUM': '#ffd700', '🟢 LOW': '#2ca02c', '✅ MINIMAL': '#1f77b4'}
colors_list = [colors_pie.get(level, '#999999') for level in risk_level_counts.index]
ax4.pie(risk_level_counts.values, labels=risk_level_counts.index, autopct='%1.1f%%', colors=colors_list, startangle=90)
ax4.set_title('Supplier Distribution by Risk Level', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Visualizations generated successfully")

## 6. Detailed Risk Analysis

In [ ]:
# Detailed analysis of high-risk suppliers
print("\n=== HIGH-RISK SUPPLIERS (Score ≥ 60) ===")
high_risk = risk_scores[risk_scores['risk_score'] >= 60].sort_values('risk_score', ascending=False)

for idx, row in high_risk.iterrows():
    print(f"\n🚨 {row['company']}")
    print(f"   Risk Score: {row['risk_score']:.1f}/100 {row['risk_level']}")
    print(f"   News Risk: {row['news_risk']:.1f} | Financial Risk: {row['financial_risk']:.1f}")
    print(f"   Recent Articles: {row['recent_articles']}")
    
    # Show relevant articles
    company_news = news_df[news_df['companies'].str.contains(row['company'], case=False, na=False)]
    if len(company_news) > 0:
        print(f"   Latest News:")
        for _, article in company_news.head(2).iterrows():
            print(f"     • {article['title'][:70]}...")
            print(f"       Sentiment: {article['sentiment']:.3f}")

In [ ]:
# Correlation analysis
print("\n=== FACTOR CORRELATIONS ===")

# Merge dataframes for correlation
analysis_df = risk_scores.copy()
if len(financial_df) > 0:
    # Calculate correlations
    print(f"\nCorrelation of Financial Metrics with Risk:")
    print(f"  Volatility → Risk: {financial_df['volatility'].corr(risk_scores['financial_risk']):.3f}")
    print(f"  Price Trend → Risk: {financial_df['price_trend'].corr(risk_scores['financial_risk']):.3f}")

print(f"\nCorrelation of News Metrics with Risk:")
print(f"  Article Count → Risk: {risk_scores['recent_articles'].corr(risk_scores['news_risk']):.3f}")

## 7. Export Results

In [ ]:
# Save results to CSV
risk_scores.to_csv('supplier_risk_assessment.csv', index=False)
print("✅ Risk assessment saved to 'supplier_risk_assessment.csv'")

# Save detailed news data
if len(news_df) > 0:
    news_df[['title', 'companies', 'published', 'sentiment']].to_csv('news_analysis.csv', index=False)
    print("✅ News analysis saved to 'news_analysis.csv'")

print("\n📊 All analysis files ready for Streamlit dashboard!")

## 8. Summary & Key Insights

In [ ]:
print("\n" + "="*60)
print("SUPPLIER RISK INTELLIGENCE - EXECUTIVE SUMMARY")
print("="*60)

print(f"\n📊 OVERVIEW:")
print(f"  Total Suppliers Analyzed: {len(risk_scores)}")
print(f"  Average Risk Score: {risk_scores['risk_score'].mean():.1f}/100")
print(f"  News Articles Processed: {len(news_df)}")
print(f"  Data Collection Period: Last 7 days")

print(f"\n🚨 CRITICAL RISKS (Score ≥ 75):")
critical = risk_scores[risk_scores['risk_score'] >= 75]
if len(critical) > 0:
    for _, row in critical.iterrows():
        print(f"  • {row['company']}: {row['risk_score']:.1f}")
else:
    print("  None - All suppliers within acceptable range")

print(f"\n⚠️  HIGH RISKS (60-75):")
high = risk_scores[(risk_scores['risk_score'] >= 60) & (risk_scores['risk_score'] < 75)]
print(f"  {len(high)} supplier(s) - Recommend monitoring")

print(f"\n✅ STABLE SUPPLIERS (< 40):")
stable = risk_scores[risk_scores['risk_score'] < 40]
print(f"  {len(stable)} supplier(s) - Low risk profile")

print(f"\n💡 KEY INSIGHTS:")
print(f"  • Average news sentiment: {news_df['sentiment'].mean():.3f}")
print(f"  • Most mentioned company: {pd.Series([c.strip() for companies in news_df['companies'].dropna() for c in companies.split(',')]).value_counts().index[0] if len(news_df) > 0 else 'N/A'}")
print(f"  • Primary risk drivers: Financial volatility & negative news sentiment")

print("\n" + "="*60)
print("✨ Ready for Streamlit Dashboard Deployment!")
print("="*60)